# UD-Pipe Evaluation on Latin Tokenisation, Lemmatisation, POS-tagging, and Processing Time
This notebook tests UD-Pipe for tokenisation, lemmatization, and POS tagging on three datasets. Results are compared to the dataset's original tags.

## Results
Running the program generates two tables for each dataset which compares the predicted results with the gold standard. The table with the suffix all_detailed includes all tokens predicted by the model, while the table with the suffix aligned_only_detailed only contains results where the model tokenised the predicted results identically to the gold_standard. Additionally, a _summary.json file will be generated that compares the precision, accuracy, recall, and f1-scores between each dataset, as well as the time required to run the model for each dataset.  


## Structure:
- Import required libraries
- Configure variables
- Import and configure model
- function word_joiner(gold_df): Convert table of gold standard words into sentences for processing
- function udpipe_processor(sample_texts, sample_type): Run model on sentences
- function assessment(merged_df): Identify the similarity between gold_df and pred_df; adds new columns indicating whether lemma and POS results match, as well as a character-level similarity score between lemmas. 
- function analyse(df, sample_type, sample_version): Analyse results and generate f-scores, precision, and accuracy scores. 
- Run program
- Save results


Created by Thea Schaaf, March 2025

## Overview
The notebook takes a csv file of an annotated gold standard (gold_df) of each data type (glosses, medieval_charters, classical_latin) and reconstructs sentences from the words in each row with the function word_joiner(). The sentences are run through the nlp model via the function udpipe_processor(), which performs the actions of tokenisation, lemmatisation, and POS-tagging on each identified token. This data is returned as a dataframe (pred_df) which is merged with gold_df to create merged_df. The function assessment() adds new columns to the dataframe that indicate whether the gold_df and pred_df lemmas and POS-tags match as well as assign a character-similarity score between the lemmas. A dataframe without token misalignment, aligned_df, is generated. Finally, the function analyse() is run on merged_df and aligned_df to generate the accuracy, precision, recall, and f1-scores for each dataframe, and the results are saved. 

## Import required libraries

In [16]:
import pandas as pd
import numpy as np
import time
import json
import os
import difflib
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Configure variables

In [3]:
# Configuration
MODEL_NAME = "UDPipe"
SAMPLE_TYPES = ["glosses", "medieval_charters", "classical_latin"]
TASKS = ["lemmatization", "pos_tagging"]

# Notebook path
notebook_path = os.path.abspath("03-udpipe.ipynb")

In [4]:
# Results storage
results = {
    "model_name": MODEL_NAME,
    "processing_times": {},
    "accuracy": {},
    "precision": {},
    "recall": {},
    "f1_score": {}
}

## Import and configure model

In [ ]:
# Import UD-pipe
from ufal.udpipe import Model, Pipeline
import ufal.udpipe

In [ ]:
# Load the UDPipe model
model_path = "/Users/Thea/Desktop/LatinNLPTools/scripts/latin-ittb-ud-2.5-191206.udpipe" # replace with own path
model = Model.load(model_path)
if not model:
    raise Exception("Model not loaded!")

In [10]:
# Create a processing pipeline
pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

## Load functions

In [5]:
def word_joiner(gold_df):
    # Extract text to reconstruct from words

    sample_texts = []
    for sample_id in gold_df['sample_id'].unique():
        words = gold_df[gold_df['sample_id'] == sample_id]['word'].tolist()
        text = ' '.join(words)
        sample_texts.append((sample_id, text))

    return sample_texts

In [6]:
def udpipe_processor(sample_texts, sample_type, pipeline):
    processed_results = []
    for sample_id, text in sample_texts:

        processed = pipeline.process(text)
        sentences = processed.strip().split('\n\n')

        for sent in sentences:
            lines = sent.split("\n")
            for idx, line in enumerate(lines):
                if line.startswith('#') or not line.strip():
                    continue

                fields = line.split('\t')
                if len(fields) != 10:
                    continue

                form = fields[1]
                lemma = fields[2]
                upos = fields[3]

                if sample_type == "glosses":
                    word_id = f"http://gams.uni-graz.at/o:glossvibe.bvi#{sample_id}.{idx - 1}"
                else:
                    word_id = str(idx)

                processed_results.append({
                    "sample_id": sample_id,
                    "word_id": word_id,
                    "word": form,
                    "lemma": lemma,
                    "pos": upos
                })

    return processed_results


In [7]:
def analyse(df, sample_type, df_version):
    # Convert the lemma columns to string type to ensure consistent comparison
    df['lemma_gold'] = df['lemma_gold'].astype(str)
    df['lemma_pred'] = df['lemma_pred'].astype(str)

    # Evaluate lemmatization
    lemma_accuracy = accuracy_score(df['lemma_gold'], df['lemma_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = df['lemma_gold'] == df['lemma_pred']

    # Fix the precision_recall_fscore_support call
    lemma_precision, lemma_recall, lemma_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(df),
        average='binary'
    )

    # Convert the lemma columns to string type to ensure consistent comparison
    df['pos_gold'] = df['pos_gold'].astype(str)
    df['pos_pred'] = df['pos_pred'].astype(str)

    # Evaluate lemmatization
    pos_accuracy = accuracy_score(df['pos_gold'], df['pos_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = df['pos_gold'] == df['pos_pred']

    # Fix the precision_recall_fscore_support call
    pos_precision, pos_recall, pos_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(df),
        average='binary'
    )

    results["accuracy"][f"{sample_type}_{df_version}_lemma"] = lemma_accuracy
    results["precision"][f"{sample_type}_{df_version}_lemma"] = lemma_precision
    results["recall"][f"{sample_type}_{df_version}_lemma"] = lemma_recall
    results["f1_score"][f"{sample_type}_{df_version}_lemma"] = lemma_f1

    results["accuracy"][f"{sample_type}_{df_version}_pos"] = pos_accuracy
    results["precision"][f"{sample_type}_{df_version}_pos"] = pos_precision
    results["recall"][f"{sample_type}_{df_version}_pos"] = pos_recall
    results["f1_score"][f"{sample_type}_{df_version}_pos"] = pos_f1

    df.to_csv(f"../results/{MODEL_NAME}_{sample_type}_{df_version}_detailed.csv", index=False)

    print(f"Completed {sample_type} {df_version}. Processing time: {processing_time:.2f}s")
    print(f"Lemmatization {df_version} accuracy: {lemma_accuracy:.4f}")
    print(f"POS tagging {df_version} accuracy: {pos_accuracy:.4f}")
    print("-" * 50)



In [8]:
def assessment(merged_df):

    # 1. POS match: exact string comparison
    merged_df['pos_match?'] = merged_df['pos_gold'] == merged_df['pos_pred']

    # 2. Lemma match: exact string comparison
    merged_df['lemma_match?'] = merged_df['lemma_gold'] == merged_df['lemma_pred']

    # 3. Lemma similarity: character-level similarity score (0 to 1)

    def lemma_similarity(a, b):
        return difflib.SequenceMatcher(None, str(a), str(b)).ratio()

    merged_df['lemma_sim?'] = merged_df.apply(
        lambda row: lemma_similarity(row['lemma_gold'], row['lemma_pred']), axis=1
    )

    return merged_df


## Run notebook

In [18]:
for sample_type in SAMPLE_TYPES:
    print(f"Processing {sample_type}...")

    # Load gold standard data
    gold_file = os.path.join(os.path.dirname(notebook_path), f"../data/gold_standard/gs_{sample_type}.csv")

    gold_df = pd.read_csv(gold_file)

    # Extract text to reconstruct from words
    sample_texts = word_joiner(gold_df)

    # Process samples and measure time
    start_time = time.time()

    processed_results = udpipe_processor(sample_texts, sample_type, pipeline)

    processing_time = time.time() - start_time
    results["processing_times"][sample_type] = processing_time

    print(f"Data processes with {MODEL_NAME} in {processing_time} seconds.")

    # Merge gold_df with processed_results
    pred_df = pd.DataFrame(processed_results)

    # Add a token index per sample in both gold and pred dataframes
    gold_df['token_idx'] = gold_df.groupby('sample_id').cumcount()
    pred_df['token_idx'] = pred_df.groupby('sample_id').cumcount()

    # Merge on sample_id and token index
    merged_df = pd.merge(gold_df, pred_df, on=['sample_id', 'token_idx'], suffixes=('_gold', '_pred'))

    # Check mismatches
    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']

    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']
    mismatched_df = merged_df[~merged_df['word_match']].head()

    merged_df = assessment(merged_df)

    # Create df version without mismatches
    aligned_df = merged_df[merged_df['word_match']].copy()

    print(f"Running analysis on {sample_type}...")
    analyse(merged_df, sample_type, "all")
    analyse(aligned_df, sample_type, "aligned_only")




Processing glosses...
Data processes with UDPipe in 0.09675121307373047 seconds.
Running analysis on glosses...
Completed glosses all. Processing time: 0.10s
Lemmatization all accuracy: 0.6832
POS tagging all accuracy: 0.6276
--------------------------------------------------
Completed glosses aligned_only. Processing time: 0.10s
Lemmatization aligned_only accuracy: 0.6884
POS tagging aligned_only accuracy: 0.6324
--------------------------------------------------
Processing medieval_charters...
Data processes with UDPipe in 4.5551369190216064 seconds.
Running analysis on medieval_charters...
Completed medieval_charters all. Processing time: 4.56s
Lemmatization all accuracy: 0.6911
POS tagging all accuracy: 0.6773
--------------------------------------------------
Completed medieval_charters aligned_only. Processing time: 4.56s
Lemmatization aligned_only accuracy: 0.7076
POS tagging aligned_only accuracy: 0.6910
--------------------------------------------------
Processing classical_la

## Save results

In [14]:
# Save summary results
with open(f"../results/{MODEL_NAME}_summary.json", "w") as f:
    json.dump(results, f, indent=2)